# Estudio del resumen de video sobre un solo video (Cap. 4)

Aplica el procedimiento de `preproc` a **un video** con `preproc_study` y muestra cada decisión: cuadros eliminados, cortes, candidatas (ventanas y tomas cortas), redundancia, miniclips y lo que recibe Swin.

**Cómo usarlo**

1. Edita la celda de **parámetros** (rutas, video, GPU).
2. Ejecuta la **configuración del entorno** una vez por sesión.
3. **Parte 1 – exploración:** cada procedimiento por separado, con gráficos en línea. Sirve para calibrar θ_L, θ_σ, θ_E, θ_H y r sin escribir archivos.
4. **Parte 2 – estudio completo:** `study.run(...)` escribe todas las salidas en `OUT` (CSV, PNG, un `.mp4` por miniclip, video anotado, `report.md`) y, con `VERIFY = True`, las compara con `preproc.process_video`.

Requisitos: entorno con GPU (Entorno de ejecución → Cambiar tipo → GPU), ffmpeg con **libx264** (Colab lo trae) y la carpeta `preprocesamiento/` con `preproc/`, `preproc_study/` y `config.yaml`.

## Parámetros

In [ ]:
from pathlib import Path

USE_DRIVE = True                                    # montar Google Drive
CODE_DIR = Path('/content/preprocesamiento')        # contiene preproc/, preproc_study/ y config.yaml
CONFIG = CODE_DIR / 'config.yaml'
TRANSNET_DIR = Path('/content/TransNetV2')          # repo clonado de TransNet V2
WEIGHTS = Path('/content/weights/TransNetV2/transnetv2-pytorch-weights.pth')

VIDEO = Path('/content/drive/MyDrive/videos/vNonPorn000006.mp4')   # video a estudiar
OUT = Path('/content/estudio') / VIDEO.stem         # salidas del estudio completo (Parte 2)
DRIVE_OUT = Path('/content/drive/MyDrive/estudio')  # dónde copiar el .tar.gz final (None = no copiar)

GPU = 0              # -1 = CPU
VERIFY = True        # Parte 2: comparar con preproc.process_video (procesa el video una segunda vez)
MAX_IMAGES = 200     # máximo de tiras por tipo (los CSV siempre están completos)

## Configuración del entorno

Monta Drive, instala `preproc` (y con él `preproc_study`) en modo editable y prepara TransNet V2 si faltan los pesos.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print('código:', CODE_DIR, CODE_DIR.is_dir())

In [ ]:
# Dependencias de preproc (numpy, scipy, PyYAML, tqdm, torch). OpenCV y matplotlib ya vienen en Colab.
!pip install -q -e {CODE_DIR}

In [ ]:
# TransNet V2: clonar el repo y convertir los pesos a PyTorch (solo si faltan; usa TensorFlow una vez).
if not WEIGHTS.exists():
    if not TRANSNET_DIR.exists():
        !git lfs install
        !git clone https://github.com/soCzech/TransNetV2 {TRANSNET_DIR}
    !cd {TRANSNET_DIR}/inference-pytorch && python convert_weights.py --test
    WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
    !cp {TRANSNET_DIR}/inference-pytorch/transnetv2-pytorch-weights.pth {WEIGHTS}
print('pesos:', WEIGHTS, WEIGHTS.exists())

In [ ]:
import torch
from preproc import check_ffmpeg, load_config

ff = check_ffmpeg()
print('ffmpeg', ff.version, '| libx264:', ff.has_libx264)
assert ff.has_libx264, 'el ffmpeg instalado no tiene libx264: hace falta para guardar los miniclips'

if GPU >= 0 and not torch.cuda.is_available():
    print('No hay GPU disponible: TransNet V2 correrá en CPU')
    GPU = -1

cfg = load_config(CONFIG)
cfg.transnet.repo_dir = str(TRANSNET_DIR)   # rutas de esta sesión, sin tocar el YAML
cfg.transnet.weights = str(WEIGHTS)

assert VIDEO.is_file(), f'no existe el video {VIDEO}'
f, s = cfg.filter, cfg.summary
print(f'θ_L = {f.luma_threshold}, θ_σ = {f.std_threshold}, θ_E = {f.static_threshold} (≥ {f.static_min_frames} cuadros)')
print(f'TransNet V2: p > {cfg.transnet.threshold}')
print(f'T = {s.frames_per_clip}, r = {s.stride}, L = {s.window_length}, θ_H = {s.hamming_threshold}, '
      f'tomas cortas = {s.short_shot_windows}')
print(f'salida: {cfg.output.size} px, {cfg.output.resize_mode}, crf {cfg.output.crf}, preset {cfg.output.preset}')

In [ ]:
import cv2
import pandas as pd
from IPython.display import Image, Video, display

from preproc_study import VideoStudy, drawing, plots
from preproc_study.export import shots_for_matrices

pd.set_option('display.max_rows', 200)


def show(image):
    """Muestra una imagen RGB uint8 (tiras y hojas de OpenCV)."""
    ok, buf = cv2.imencode('.png', cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    display(Image(data=buf.tobytes()))


study = VideoStudy(VIDEO, cfg, gpu=GPU, ff=ff)

# Parte 1 – Exploración procedimiento por procedimiento

Nada de esta parte escribe archivos. Cada método guarda su resultado en `study` (`study.decoded`, `study.filtering`, …) y acepta umbrales con nombre que valen solo para esa llamada.

## 1. Decodificación a 25 fps

In [ ]:
decoded = study.decode()
meta = decoded.metadata()
display(pd.Series(meta['source'], name='original'))
display(pd.Series(meta['timeline'], name='línea a 25 fps'))
if meta['mapping']:
    display(pd.Series(meta['mapping'], name='correspondencia estimada'))
for w in decoded.warnings:
    print('aviso:', w)

## 2. Filtrado de cuadros negros, uniformes y pantallas fijas

In [ ]:
filtering = study.filter_frames()                   # umbrales del YAML
display(plots.plot_frame_filtering(filtering, decoded.fps))
display(pd.Series(filtering.counts(), name='cuadros'))
pd.DataFrame([{'run': r.index, 'start_s': round(r.start / decoded.fps, 2), 'duration_s': round(r.length / decoded.fps, 2),
               'reason': r.reason} for r in filtering.runs])

In [ ]:
# Prueba con otros umbrales (no cambia el YAML ni study.config).
# Ojo: deja este resultado en study.filtering; vuelve a ejecutar la celda anterior para regresar a los del YAML.
trial = study.filter_frames(luma_threshold=20, static_threshold=0.8)
display(plots.plot_frame_filtering(trial, decoded.fps))
filtering = study.filter_frames()

## 3. Tomas (TransNet V2) y candidatas

Ventanas de L = T·r cuadros en las tomas de ≥ L cuadros y, con `short_shot_windows`, una candidata de T cuadros consecutivos (r = 1) en cada toma de T a L − 1 cuadros. TransNet V2 se ejecuta una sola vez mientras los segmentos no cambien.

In [ ]:
shots = study.detect_shots()
display(plots.plot_transition_probabilities(shots, decoded.fps))
display(plots.plot_shot_lengths(shots, decoded.fps))
pd.Series(shots.counts(), name='tomas y candidatas')

In [ ]:
# Transiciones más dudosas (pico de p más cercano al umbral) con sus cuadros.
doubtful = sorted(shots.transitions, key=lambda t: t.peak)[:3]
frames_needed = {n for t in doubtful for _, n in drawing.transition_frames(t)}
thumbs = study.read_frames(sorted(n for n in frames_needed if 0 <= n < decoded.n_frames), width=160)
size = next(iter(thumbs.values())).shape[1::-1] if thumbs else (160, 90)
for t in doubtful:
    show(drawing.transition_strip(t, shots, thumbs, size, decoded.fps))

In [ ]:
# Barrido de r: cuántas tomas dan ventana, cuántas dan candidata de toma corta y cuántas ninguna.
pd.DataFrame(study.sweep_stride((1, 2, 4, 8)))

## 4. Eliminación de redundancia por pHash (θ_H)

Solo se comparan las ventanas de una misma toma; las candidatas de tomas cortas se conservan siempre. **θ_H es el único control de cuántos miniclips da el video.**

In [ ]:
redundancy = study.remove_redundancy()
display(pd.Series(redundancy.counts(), name='ventanas'))

sweep = study.sweep_hamming_threshold(range(0, 33, 2))
display(plots.plot_hamming_threshold_sweep(sweep, current=redundancy.threshold))
pd.DataFrame(sweep)

In [ ]:
# Matriz de distancias de la toma con más ventanas y las ventanas descartadas junto a su referencia.
for shot in shots_for_matrices(redundancy, 1):
    display(plots.plot_distance_matrix(redundancy, shot))

discarded = redundancy.discarded[:3]
needed = sorted({n for d in discarded for g in drawing.discarded_window_frames(d, redundancy) for n in g})
thumbs = study.read_frames(needed, width=160) if needed else {}
size = next(iter(thumbs.values())).shape[1::-1] if thumbs else (160, 90)
for d in discarded:
    show(drawing.discarded_window_strip(d, redundancy, thumbs, size, decoded.fps))

## 5. Miniclips

Toda candidata conservada es un miniclip, en orden temporal. Si no queda ninguna (N = 0), un único miniclip de la toma más larga.

In [ ]:
miniclips = study.plan_miniclips()
display(pd.Series(miniclips.counts(), name='miniclips'))
pd.DataFrame(miniclips.rows())

In [ ]:
# Los 32 cuadros que recibe Swin de un miniclip (sin escribir nada).
K = 0
frames = study.read_miniclip(K)                     # (32, 224, 224, 3) uint8
clip = miniclips.miniclips[K]
show(drawing.miniclip_sheet(frames, clip, K, cfg.fps, miniclips.stride(K), miniclips.playback_fps(K)))

In [ ]:
timeline = study.timeline()
plots.plot_timeline(timeline, shots, miniclips)

## 6. Calibración de θ_H

Prueba un valor, mira cuántos miniclips resultan y, si te convence, déjalo en `study.config` para la Parte 2. El YAML no se modifica: cuando elijas el valor definitivo, cámbialo en `config.yaml`.

In [ ]:
THETA_H = cfg.summary.hamming_threshold    # cambia este valor para probar

trial_redundancy = study.remove_redundancy(hamming_threshold=THETA_H)
trial_miniclips = study.plan_miniclips(shots, trial_redundancy)
print(f'θ_H = {THETA_H}: {trial_redundancy.counts()}')
display(pd.Series(trial_miniclips.counts(), name=f'θ_H = {THETA_H}'))

APPLY = False                              # True: la Parte 2 usará este θ_H
if APPLY:
    study.config.summary.hamming_threshold = float(THETA_H)
print('θ_H de la Parte 2:', study.config.summary.hamming_threshold)

# Parte 2 – Estudio completo

Recalcula todo con `study.config` y escribe las salidas en `OUT`. Los miniclips se procesan uno por uno: cada uno se extrae, se guarda como `.mp4` (igual que `preproc`), se vuelve a leer para medir el PSNR del almacenamiento y se libera.

In [ ]:
summary = study.run(
    OUT,
    annotated_video=True,
    max_images=MAX_IMAGES,
    strides=(1, 2, 4, 8),
    hamming_thresholds=range(0, 33, 2),
    sheets=True,             # hoja PNG con los 32 cuadros de cada miniclip
    measure_storage=True,    # PSNR del .mp4 y estadísticas de lo que recibe Swin
    verify=VERIFY,
)
print('salidas en', OUT)
for w in summary['warnings']:
    print('aviso:', w)
pd.Series(summary['timings_s'], name='segundos')

In [ ]:
display(pd.Series(summary['miniclips'], name='miniclips'))
swin = summary['swin_input']
display(pd.Series(swin['storage'], name='almacenamiento'))
pd.DataFrame({'before': swin['before_normalization'], 'after': swin['after_normalization']})

In [ ]:
# Comparación con preproc.process_video (solo con VERIFY = True): todas las filas deberían coincidir.
if summary['verification'] is not None:
    print('todo coincide' if summary['verification']['all_match'] else 'HAY DIFERENCIAS')
    display(pd.DataFrame(summary['verification']['rows'])[['item', 'match', 'study', 'preproc']])

In [ ]:
display(Image(filename=str(OUT / 'timeline.png')))
display(Image(filename=str(OUT / '04_redundancy_removal' / 'hamming_threshold_sweep.png')))
sheet = OUT / '06_swin_input' / 'miniclip_000.png'
if sheet.exists():
    display(Image(filename=str(sheet)))

In [ ]:
# Un miniclip tal como lo guarda preproc (a 25 / r fps) y el video anotado (puede pesar en videos largos).
display(Video(str(OUT / '06_swin_input' / 'miniclip_000.mp4'), embed=True, width=224))
SHOW_ANNOTATED = False
if SHOW_ANNOTATED:
    display(Video(str(OUT / 'annotated_video.mp4'), embed=True, width=480))

In [ ]:
# Lectura de un miniclip guardado y tensor que recibe Video Swin.
from preproc import load_miniclip, normalize, to_cthw

stored = load_miniclip(OUT / '06_swin_input' / 'miniclip_000.mp4', ff)   # (32, 224, 224, 3) uint8
x = to_cthw(normalize(stored, cfg.output.mean, cfg.output.std))          # (3, 32, 224, 224) float32
print(stored.shape, stored.dtype, '->', x.shape, x.dtype)

In [ ]:
# El informe completo con enlaces a todas las salidas.
from IPython.display import Markdown
Markdown((OUT / 'report.md').read_text(encoding='utf-8'))

## Copiar las salidas a Drive

Un solo `.tar.gz` por video: copiar muchos archivos sueltos a Drive es lento.

In [ ]:
import shutil

if DRIVE_OUT is not None:
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    archive = shutil.make_archive(str(DRIVE_OUT / OUT.name), 'gztar', root_dir=OUT.parent, base_dir=OUT.name)
    print('copiado a', archive)